**This cell just sets up imports, file paths, and global knobs you may want to tweak.**


In [ ]:
import os, glob, math, time, re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

#Point these search roots to wherever your model grids live.
SEARCH_ROOTS = [
    "/Users/carlimankowski/research/grids_real",
    "/Users/carlimankowski/research/grids",
]
#STAR_FILE is your input catalog
STAR_FILE = "clusterwork.csv"
OUT_FILE  = "clusterwork_AGES_FIXED.csv"

NU_MAX_SUN = 3090.0  # μHz
DNU_SUN    = 135.1   # μHz
TEFF_SUN   = 5772.0  # K
LOGG_SUN   = 4.438

#These weights control how strongly each observable pulls on the fit.
W_MASS   = 8.0    
W_RADIUS = 0.0   
W_DENS   = 1.0    
W_LOGG   = 0.0   
W_TEFF   = 0.2  
W_MET    = 0.05  
W_LOGG_SEIS = 0.5 

# Propagation/systematic floors
SIGMA_T_FLOOR   = 80.0  
SIGMA_MET_FLOOR = 0.10  
FRAC_SYS_NUMAX  = 0.00  
FRAC_SYS_DNU    = 0.00 
SIGMA_LOGG_FLOOR= 0.10   
FRAC_M_FLOOR    = 0.06   

# Proposal (speed/robustness) — proposal ONLY, not scored in χ²
APPLY_MASS_PREFILTER   = True
MASS_WINDOW_K_SEQUENCE = [2.0, 4.0, 8.0, 16.0]
TOP_PER_GRID           = 3000

# Proposal age window derived from seismic mass (honest! no clamp)
USE_AGE_BAND_FROM_MASS = True
AGE_PROP_WIDTH_FRAC    = 0.50  
TAU_MS_EXPONENT        = 3.2

#Tiny helper utilities used throughout the notebook.
_num = lambda x: pd.to_numeric(x, errors="coerce")

def _nanmean_pair(a, b):
    a = float(a) if pd.notna(a) else np.nan
    b = float(b) if pd.notna(b) else np.nan
    if np.isfinite(a) and np.isfinite(b): return 0.5 * (abs(a) + abs(b))
    if np.isfinite(a): return abs(a)
    if np.isfinite(b): return abs(b)
    return np.nan

def _norm(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")


**This cell figures out where your grids are and how to read their columns.**

In [ ]:
def _discover_parquets():
    # now look for 4 families: MIST, Dartmouth, GARSTEC, YREC
    fams = {"MIST": [], "Dartmouth": [], "GARSTEC": [], "YREC": []}
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for pat in ("*.parquet", "*.pqt"):
            for p in glob.glob(os.path.join(root, "**", pat), recursive=True):
                pl = p.lower()
                if "mist" in pl:
                    fam = "MIST"
                elif "dartmouth" in pl:
                    fam = "Dartmouth"
                elif "garstec" in pl:
                    fam = "GARSTEC"
                elif "yrec" in pl:
                    fam = "YREC"
                else:
                    fam = None
                if fam:
                    fams[fam].append(p)

    chosen = {}
    for fam, paths in fams.items():
        if not paths:
            continue
        # prefer EEP and larger files
        paths.sort(
            key=lambda p: (("eep" in p.lower()),
                           os.path.getsize(p) if os.path.exists(p) else 0),
            reverse=True,
        )
        chosen[fam] = paths[0]

    # require all four
    missing = [f for f in ("MIST", "Dartmouth", "GARSTEC", "YREC") if f not in chosen]
    if missing:
        raise FileNotFoundError(f"Missing Parquet for {missing}. Searched {SEARCH_ROOTS}")
    return chosen


def _peek_names(pf: pq.ParquetFile):
    tbl = pf.read_row_groups([0])
    cols = list(tbl.to_pandas().columns)
    nmap = {_norm(c): c for c in cols}
    return cols, nmap

def _pick(nmap, exact_lists, contains_lists=None):
    for names in exact_lists:
        for nm in names:
            if nm in nmap:
                return nmap[nm]
    if contains_lists:
        for subs in contains_lists:
            for k, orig in nmap.items():
                if all(s in k for s in subs):
                    return orig
    return None

def _column_mapping(cols, nmap):
    age = _pick(
        nmap,
        [[_norm("star_age")], [_norm("age_gyr")], [_norm("age_yr")],
         [_norm("age_myr")], [_norm("log10_age_yr")], [_norm("age")]],
        [["age", "gyr"], ["age", "yr"], ["age", "myr"], ["log", "age", "yr"]],
    )
    mass = _pick(
        nmap,
        [[_norm("star_mass")], [_norm("current_mass")], [_norm("mass")],
         [_norm("mass_msun")], [_norm("initial_mass")]],
        [["star", "mass"], ["current", "mass"], ["mass"]],
    )
    numax_col = _pick(
        nmap,
        [[_norm("nu_max")], [_norm("numax")], [_norm("nu_max_uhz")]],
        [["nu", "max"]],
    )
    dnu_col = _pick(
        nmap,
        [[_norm("delta_nu")], [_norm("dnu")], [_norm("delta_nu_uhz")]],
        [["delta", "nu"], ["dnu"]],
    )
    teff = _pick(
        nmap,
        [[_norm("log_teff")], [_norm("log_t")], [_norm("teff_k")], [_norm("teff")]],
        [["log", "teff"], ["log", "t"], ["teff"]],
    )
    logR = _pick(nmap, [[_norm("log_r")], [_norm("log_radius")]], [["log", "r"]])
    logg = _pick(
        nmap,
        [[_norm("log_g")], [_norm("logg")], [_norm("logg_cgs")], [_norm("surface_gravity")]],
        [["log", "g"]],
    )
    logL = _pick(
        nmap,
        [[_norm("log_l")], [_norm("log_lum")], [_norm("log_lbol")], [_norm("log_l_div_ledd")]],
        [["log", "l"]],
    )
    met = _pick(
        nmap,
        [[_norm("fe_h")], [_norm("[fe/h]")], [_norm("initial_[fe/h]")],
         [_norm("surface_[fe/h]")], [_norm("feh")], [_norm("metallicity")],
         [_norm("initial_metallicity")]],
        [["fe", "h"], ["met"]],
    )

    have_geom = (teff is not None) and (
        (logR is not None) or (logL is not None) or (logg is not None)
    )
    if not (age and have_geom):
        return None

    return {
        "age": age,
        "mass": mass,
        "numax": numax_col,
        "dnu": dnu_col,
        "teff": teff,
        "logR": logR,
        "logg": logg,
        "logL": logL,
        "met": met,
    }

def _age_to_gyr(series, name):
    s = _num(series)
    nm = _norm(name)
    if "log10_age_yr" in nm:
        return (10.0 ** s) / 1e9
    if "age_myr" in nm:
        return s / 1e3
    if "age_yr" in nm or "star_age" in nm:
        med = np.nanmedian(s)
        return s / 1e9 if med > 1e6 else (s / 1e3 if med > 100 else s)
    med = np.nanmedian(s)
    if med > 1e6:
        return s / 1e9
    if med > 100:
        return s / 1e3
    return s

def _iter_rowgroups(pf, usecols):
    nrg = pf.metadata.num_row_groups
    for i in range(nrg):
        tbl = pf.read_row_groups([i], columns=usecols)
        yield tbl.to_pandas()


**This cell contains the asteroseismic scaling and the main per-grid fitters.**

In [ ]:
def tau_ms_from_mass(M, expo=TAU_MS_EXPONENT):
    # main-sequence lifetime in Gyr
    return 10.0 * (M ** (-expo))

def _seismic_summary(nu, s_nu, dnu, s_dnu, Teff, s_T,
                     c_numax=1.0, c_dnu=1.0, f_dnu=1.0):
    # corrections
    nu_c  = c_numax * nu
    dnu_c = c_dnu * f_dnu * dnu
    # fractional errors with systematics
    f_nu = np.hypot(s_nu / max(abs(nu), 1e-12), FRAC_SYS_NUMAX)
    f_d  = np.hypot(s_dnu / max(abs(dnu), 1e-12), FRAC_SYS_DNU)
    f_T  = s_T / max(abs(Teff), 1e-12)

    # Mass scaling
    M = (nu_c / NU_MAX_SUN) ** 3 * (dnu_c / DNU_SUN) ** (-4) * (Teff / TEFF_SUN) ** 1.5
    sigma_lnM = np.sqrt((3 * f_nu) ** 2 + (4 * f_d) ** 2 + (1.5 * f_T) ** 2)
    sM = abs(M) * sigma_lnM

    # Radius scaling 
    R = (nu_c / NU_MAX_SUN) * (dnu_c / DNU_SUN) ** (-2) * (Teff / TEFF_SUN) ** 0.5
    sigma_lnR = np.sqrt((1 * f_nu) ** 2 + (2 * f_d) ** 2 + (0.5 * f_T) ** 2)
    sR = abs(R) * sigma_lnR

    # Mean density scaling 
    rho   = (dnu_c / DNU_SUN) ** 2
    s_rho = abs(rho) * np.sqrt((2 * f_d) ** 2)
    return M, sM, R, sR, rho, s_rho

def _logg_seismic(nu, s_nu, Teff, s_T, c_numax=1.0):
    nu_c = c_numax * nu
    lg = LOGG_SUN + math.log10((nu_c / NU_MAX_SUN) * (Teff / TEFF_SUN) ** 0.5)
    f_nu = s_nu / max(abs(nu), 1e-12)
    f_T  = s_T / max(abs(Teff), 1e-12)
    sig_log10 = (1 / np.log(10)) * np.hypot(f_nu, 0.5 * f_T)
    return lg, max(sig_log10, 0.03)  

def _huber(residual, sigma, delta=2.0):
    z = np.abs(residual) / max(sigma, 1e-12)
    return np.where(z <= delta, 0.5 * z * z, delta * (z - 0.5 * delta))

def _fit_star_one_grid_masscentric(
    pf, cm,
    M_seis, sM_eff, R_seis, sR, rho_seis, s_rho,
    lg_o, s_lg, teff_o, s_T, feh_o, s_feh,
    lg_seis, s_lg_seis,
    mass_center, sM_prop, age_band, kM,
    apply_mass_prefilter=True
):
    need = [cm["age"]]
    for c in [cm["mass"], cm["logR"], cm["logL"], cm["logg"], cm["teff"], cm["met"]]:
        if c:
            need.append(c)
    need = list(dict.fromkeys(need))

    kept = []
    scanned = 0
    t0 = time.time()

    # can we even try to do a mass-based window?
    can_mass = (
        (cm["mass"] in pf.schema.names)
        or (cm["logg"] in pf.schema.names and ((cm["logR"] in pf.schema.names) or (cm["logL"] in pf.schema.names)))
    )
    lowM, highM = mass_center - kM * sM_prop, mass_center + kM * sM_prop

    for chunk in _iter_rowgroups(pf, usecols=need):
        scanned += len(chunk)
        msk = pd.Series(True, index=chunk.index)

        # proposal age window 
        if age_band is not None:
            ag = _age_to_gyr(chunk[cm["age"]], cm["age"])
            msk &= _num(ag).between(age_band[0], age_band[1])

        # proposal mass window
        if apply_mass_prefilter and can_mass:
            if (cm["mass"] is not None) and (cm["mass"] in chunk.columns):
                M_pf = _num(chunk[cm["mass"]])
            else:
                R_pf = None
                if (cm["logR"] is not None) and (cm["logR"] in chunk.columns):
                    R_pf = 10.0 ** _num(chunk[cm["logR"]])
                elif (cm["logL"] is not None) and (cm["logL"] in chunk.columns) and (cm["teff"] in chunk.columns):
                    Teff_pf = (
                        10.0 ** _num(chunk[cm["teff"]])
                        if "log" in _norm(cm["teff"])
                        else _num(chunk[cm["teff"]])
                    )
                    L_pf = 10.0 ** _num(chunk[cm["logL"]])
                    R_pf = np.sqrt(L_pf / ((Teff_pf / TEFF_SUN) ** 4))

                if (cm["logg"] is not None) and (cm["logg"] in chunk.columns) and (R_pf is not None):
                    lg_pf = _num(chunk[cm["logg"]])
                    M_pf  = (10.0 ** (lg_pf - LOGG_SUN)) * (R_pf ** 2)
                else:
                    M_pf = None

            if M_pf is not None:
                msk &= _num(M_pf).between(lowM, highM)

        sub = chunk.loc[msk]
        if sub.empty:
            continue

        age_gyr = _age_to_gyr(sub[cm["age"]], cm["age"])

        # model mass
        if (cm["mass"] is not None) and (cm["mass"] in sub.columns):
            M_mod = _num(sub[cm["mass"]])
        else:
            R_m  = 10.0 ** _num(sub[cm["logR"]]) if (cm["logR"] in sub.columns) else np.nan
            lg_m = _num(sub[cm["logg"]]) if (cm["logg"] in sub.columns) else np.nan
            M_mod = (10.0 ** (lg_m - LOGG_SUN)) * (R_m ** 2)

        # model R, density
        R_m     = 10.0 ** _num(sub[cm["logR"]]) if (cm["logR"] in sub.columns) else np.nan
        rho_mod = (M_mod / (R_m ** 3)) if (cm["logR"] in sub.columns) else np.nan

        chi2 = pd.Series(0.0, index=sub.index)

        # If mass is all NaN for this grid, skip mass/density terms entirely
        finite_M = np.isfinite(M_mod)
        if finite_M.any():
            chi2 += W_MASS * _huber(M_mod - M_seis, sM_eff)
            if np.isfinite(rho_seis) and np.isfinite(s_rho) and (cm["logR"] in sub.columns) and (W_DENS > 0):
                chi2 += W_DENS * _huber(rho_mod - rho_seis, s_rho)

        if np.isfinite(R_seis) and np.isfinite(sR) and (cm["logR"] in sub.columns) and (W_RADIUS > 0):
            chi2 += W_RADIUS * _huber(R_m - R_seis, sR)

        if (cm["logg"] in sub.columns) and np.isfinite(lg_seis) and (s_lg_seis > 0) and (W_LOGG_SEIS > 0):
            lg_m = _num(sub[cm["logg"]])
            chi2 += W_LOGG_SEIS * ((lg_m - lg_seis) ** 2) / (s_lg_seis ** 2)

        if (W_TEFF > 0) and np.isfinite(teff_o) and np.isfinite(s_T) and (s_T > 0) and (cm["teff"] in sub.columns):
            teff_name   = cm["teff"]
            is_log_teff = "log" in _norm(teff_name)
            teff_m = 10.0 ** _num(sub[teff_name]) if is_log_teff else _num(sub[teff_name])
            chi2 += W_TEFF * ((teff_m - teff_o) ** 2) / (s_T ** 2)

        if (W_MET > 0) and np.isfinite(feh_o) and np.isfinite(s_feh) and (s_feh > 0) and (cm["met"] in sub.columns):
            met_m = _num(sub[cm["met"]])
            chi2 += W_MET * ((met_m - feh_o) ** 2) / (s_feh ** 2)

        preds = pd.DataFrame({"age_gyr": age_gyr, "chi2": chi2})
        preds.replace([np.inf, -np.inf], np.nan, inplace=True)
        preds = preds.dropna()
        if preds.empty:
            continue

        if len(preds) > TOP_PER_GRID * 2:
            preds = preds.nsmallest(TOP_PER_GRID * 2, "chi2")

        kept.append(preds)

        if sum(len(x) for x in kept) >= TOP_PER_GRID * 6:
            break

    if not kept:
        return None, scanned, time.time() - t0

    pool = pd.concat(kept, ignore_index=True)
    best = pool.nsmallest(TOP_PER_GRID, "chi2")
    return best, scanned, time.time() - t0

# --- GARSTEC-only SIMPLE fallback: Teff + seismic logg only (no mass/density) ---
def _fit_star_garstec_simple(
    pf, cm,
    teff_o, s_T,
    lg_seis, s_lg_seis
):
    need = [cm["age"]]
    if cm["teff"]: need.append(cm["teff"])
    if cm["logg"]: need.append(cm["logg"])
    need = list(dict.fromkeys(need))

    kept = []
    scanned = 0
    t0 = time.time()

    for chunk in _iter_rowgroups(pf, usecols=need):
        scanned += len(chunk)
        age_gyr = _age_to_gyr(chunk[cm["age"]], cm["age"])
        chi2 = pd.Series(0.0, index=chunk.index)

        # Teff term
        if (cm["teff"] in chunk.columns) and np.isfinite(teff_o) and (s_T > 0):
            tname = cm["teff"]
            is_log = "log" in _norm(tname)
            teff_m = 10.0 ** _num(chunk[tname]) if is_log else _num(chunk[tname])
            chi2 += ((teff_m - teff_o) ** 2) / (s_T ** 2)

        # seismic logg term
        if (cm["logg"] in chunk.columns) and np.isfinite(lg_seis) and (s_lg_seis > 0):
            lg_m = _num(chunk[cm["logg"]])
            chi2 += ((lg_m - lg_seis) ** 2) / (s_lg_seis ** 2)

        preds = pd.DataFrame({"age_gyr": age_gyr, "chi2": chi2})
        preds.replace([np.inf, -np.inf], np.nan, inplace=True)
        preds = preds.dropna()
        if preds.empty:
            continue

        kept.append(preds)

    if not kept:
        return None, scanned, time.time() - t0

    pool = pd.concat(kept, ignore_index=True)
    # i don't care about TOP_PER_GRID here; just get a sane posterior-ish mean
    chi2_arr = pool["chi2"].to_numpy()
    chi2_min = np.nanmin(chi2_arr)
    wts = np.exp(-0.5 * (chi2_arr - chi2_min))
    if not (np.isfinite(wts).any() and (wts.sum() > 0)):
        # flat weights if everything is weird
        wts = np.ones_like(chi2_arr)
    mu = float((wts * pool["age_gyr"].to_numpy()).sum() / wts.sum())
    return mu, scanned, time.time() - t0


**This cell loads your input CSV and computes per-star seismic masses etc**

In [ ]:
if not os.path.exists(STAR_FILE):
    raise FileNotFoundError(STAR_FILE)

stars = pd.read_csv(STAR_FILE)

req = [
    "TICID", "numax_corr", "numax_error_upper", "numax_error_lower",
    "dnu_corr", "dnu_error_upper", "dnu_error_lower",
]
miss = [c for c in req if c not in stars.columns]
if miss:
    raise ValueError(f"{STAR_FILE} missing: {miss}")

stars["nu_use"]  = _num(stars["numax_corr"])
stars["dnu_use"] = _num(stars["dnu_corr"])
stars["s_nu"] = [
    _nanmean_pair(u, l) for u, l in zip(stars["numax_error_upper"], stars["numax_error_lower"])
]
stars["s_dnu"] = [
    _nanmean_pair(u, l) for u, l in zip(stars["dnu_error_upper"], stars["dnu_error_lower"])
]

stars.loc[~np.isfinite(stars["s_nu"])  | (stars["s_nu"]  <= 0), "s_nu"]  = 0.05 * stars["nu_use"].abs()
stars.loc[~np.isfinite(stars["s_dnu"]) | (stars["s_dnu"] <= 0), "s_dnu"] = 0.01 * stars["dnu_use"].abs()

# log g (optional — ignored in χ², but we keep errors around)
use_logg = all(c in stars.columns for c in ["cat_logg", "cat_logg_err_upper", "cat_logg_err_lower"])
if use_logg:
    stars["lg_use"] = _num(stars["cat_logg"])
    stars["s_lg"] = [
        _nanmean_pair(u, l) for u, l in zip(stars["cat_logg_err_upper"], stars["cat_logg_err_lower"])
    ]
else:
    stars["lg_use"] = np.nan
    stars["s_lg"]   = np.nan

stars.loc[~np.isfinite(stars["s_lg"]) | (stars["s_lg"] <= 0), "s_lg"] = SIGMA_LOGG_FLOOR

# Teff 
tcol = next((c for c in ["cat_teff", "teff_k", "Teff", "teff"] if c in stars.columns), None)
stars["teff_use"] = _num(stars[tcol]) if tcol else TEFF_SUN
stars.loc[~np.isfinite(stars["teff_use"]) | (stars["teff_use"] <= 0), "teff_use"] = TEFF_SUN

t_err_candidates = [
    c for c in ["teff_err", "cat_teff_err", "teff_error", "teff_err_upper", "teff_err_lower"]
    if c in stars.columns
]
if {"teff_err_upper", "teff_err_lower"}.issubset(stars.columns):
    stars["s_T"] = [
        _nanmean_pair(u, l) for u, l in zip(stars["teff_err_upper"], stars["teff_err_lower"])
    ]
elif t_err_candidates:
    stars["s_T"] = _num(stars[t_err_candidates[0]])
else:
    stars["s_T"] = SIGMA_T_FLOOR

stars.loc[~np.isfinite(stars["s_T"]) | (stars["s_T"] <= 0), "s_T"] = SIGMA_T_FLOOR

# met
use_met = False
feh_col = next((c for c in stars.columns if "feh" in _norm(c) or "met" in _norm(c)), None)
if feh_col:
    stars["feh_use"] = _num(stars[feh_col])
    err_upper = next(
        (ec for ec in [feh_col + "_err_upper", "cat_feh_err_upper"] if ec in stars.columns),
        None,
    )
    err_lower = next(
        (ec for ec in [feh_col + "_err_lower", "cat_feh_err_lower"] if ec in stars.columns),
        None,
    )
    err_single = next(
        (ec for ec in [feh_col + "_err", "cat_feh_err"] if ec in stars.columns),
        None,
    )

    if err_upper and err_lower:
        stars["s_feh"] = [
            _nanmean_pair(u, l) for u, l in zip(stars[err_upper], stars[err_lower])
        ]
    elif err_single:
        stars["s_feh"] = _num(stars[err_single])
    else:
        stars["s_feh"] = SIGMA_MET_FLOOR

    stars.loc[~np.isfinite(stars["s_feh"]) | (stars["s_feh"] <= 0), "s_feh"] = SIGMA_MET_FLOOR
    use_met = True

# usable rows
mok = (
    stars["nu_use"].gt(0) & stars["nu_use"].notna()
    & stars["dnu_use"].gt(0) & stars["dnu_use"].notna()
)
stars_use = stars.loc[mok].copy().reset_index(drop=True)
print(f"Stars usable: {len(stars_use)}/{len(stars)}")

# per-star 
def _get_scale(row, key, default=1.0):
    v = row.get(key, default)
    try:
        return float(v) if pd.notna(v) else default
    except Exception:
        return default

M_seis_arr   = []
sM_eff_arr   = []
R_seis_arr   = []
sR_arr       = []
rho_arr      = []
s_rho_arr    = []
lg_seis_arr  = []
s_lg_seis_arr= []

for i, r in stars_use.iterrows():
    fdnu = _get_scale(r, "fdnu_used",  1.0)
    cnu  = _get_scale(r, "C_NUMAX",    1.0)
    cdn  = _get_scale(r, "C_DNU",      1.0)

    M_seis, sM, R_seis, sR, rho_seis, s_rho = _seismic_summary(
        nu=float(r["nu_use"]),   s_nu=float(r["s_nu"]),
        dnu=float(r["dnu_use"]), s_dnu=float(r["s_dnu"]),
        Teff=float(r["teff_use"]), s_T=float(r["s_T"]),
        c_numax=cnu, c_dnu=cdn, f_dnu=fdnu,
    )
    lg_seis, s_lg_seis = _logg_seismic(
        r["nu_use"], r["s_nu"], r["teff_use"], r["s_T"], cnu
    )

    sM_eff = max(sM, FRAC_M_FLOOR * abs(M_seis))

    M_seis_arr.append(M_seis)
    sM_eff_arr.append(sM_eff)
    R_seis_arr.append(R_seis)
    sR_arr.append(max(sR, 1e-4))
    rho_arr.append(rho_seis)
    s_rho_arr.append(max(s_rho, 1e-6))
    lg_seis_arr.append(lg_seis)
    s_lg_seis_arr.append(max(s_lg_seis, 0.03))

stars_use["M_seis"]    = np.array(M_seis_arr)
stars_use["sM_eff"]    = np.array(sM_eff_arr)
stars_use["R_seis"]    = np.array(R_seis_arr)
stars_use["sR"]        = np.array(sR_arr)
stars_use["rho_seis"]  = np.array(rho_arr)
stars_use["s_rho"]     = np.array(s_rho_arr)
stars_use["lg_seis"]   = np.array(lg_seis_arr)
stars_use["s_lg_seis"] = np.array(s_lg_seis_arr)

# detect cluster column 
cluster_col = next((c for c in stars_use.columns if "cluster" in _norm(c)), None)
if cluster_col:
    print(f"Cluster column detected (no pooling used): {cluster_col}")
else:
    print("No cluster column found; using individual seismic masses.")


**This final cell runs the fit for every star and writes the ages to disk.**

In [ ]:
chosen = _discover_parquets()
print("Using Parquet grids:")
for fam, p in chosen.items():
    try:
        size_mb = os.path.getsize(p) / 1e6
    except Exception:
        size_mb = float("nan")
    print(f" - {fam}: {p} ({size_mb:.1f} MB)")

parq = {fam: pq.ParquetFile(p) for fam, p in chosen.items()}

colmaps = {}
for fam, pf in parq.items():
    cols, nmap = _peek_names(pf)
    cm = _column_mapping(cols, nmap)
    if cm is None:
        print(f"[{fam}] Columns available:\n {cols}")
        raise RuntimeError(
            f"[{fam}] Could not map: need age and geometry (teff & (logR|logg|logL))."
        )
    colmaps[fam] = cm
    print(
        f"[{fam}] cols: age={cm['age']} mass={cm['mass']} numax={cm['numax']} dnu={cm['dnu']} "
        f"teff={cm['teff']} logR={cm['logR']} logg={cm['logg']} logL={cm['logL']} met={cm['met']}"
    )

rows = []

for i in range(len(stars_use)):
    r = stars_use.iloc[i]

    # per-star seismic mass and error 
    M_seis = float(r["M_seis"])
    sM_eff = float(r["sM_eff"])

    R_seis   = float(r["R_seis"])
    sR       = float(r["sR"])
    rho_seis = float(r["rho_seis"])
    s_rho    = float(r["s_rho"])
    lg_o     = float(r["lg_use"]) if pd.notna(r.get("lg_use")) else np.nan
    s_lg     = float(r["s_lg"])   if pd.notna(r.get("s_lg"))   else np.nan
    teff_o   = float(r["teff_use"])
    s_T      = float(r["s_T"])
    feh_o    = float(r["feh_use"]) if use_met and pd.notna(r.get("feh_use")) else np.nan
    s_feh    = float(r["s_feh"])   if use_met and pd.notna(r.get("s_feh"))   else np.nan
    lg_seis  = float(r["lg_seis"])
    s_lg_seis= float(r["s_lg_seis"])

    # proposal-only age band
    if USE_AGE_BAND_FROM_MASS and np.isfinite(M_seis) and (M_seis > 0):
        tau = tau_ms_from_mass(M_seis)  
        w   = AGE_PROP_WIDTH_FRAC * tau
        calc_low, calc_high = max(0.05, tau - w), min(14.0, tau + w)
        desired_low, desired_high = 0.1, 13.5
        lower, upper = max(calc_low, desired_low), min(calc_high, desired_high)
        if lower > upper:
            lower, upper = desired_low, desired_high
        band_use = (lower, upper)
    else:
        band_use = None

    success      = False
    per_grid_mu  = {}
    pool         = []
    grid_success = {fam: False for fam in parq.keys()}

    for kM in MASS_WINDOW_K_SEQUENCE:
        for fam, pf in parq.items():
            cm = colmaps[fam]

            if grid_success[fam]:
                continue

            best, scanned, dt = _fit_star_one_grid_masscentric(
                pf, cm,
                M_seis=M_seis, sM_eff=sM_eff,
                R_seis=R_seis, sR=sR,
                rho_seis=rho_seis, s_rho=s_rho,
                lg_o=lg_o, s_lg=s_lg,
                teff_o=teff_o, s_T=s_T,
                feh_o=feh_o, s_feh=s_feh,
                lg_seis=lg_seis, s_lg_seis=s_lg_seis,
                mass_center=M_seis, sM_prop=max(sM_eff, 0.10),
                age_band=band_use, kM=kM,
                apply_mass_prefilter=APPLY_MASS_PREFILTER,
            )

            kept = 0 if best is None else len(best)
            print(
                f"[{fam}] row {i} M={M_seis:.3f}±{sM_eff:.3f} "
                f"k={kM:.0f} scanned≈{scanned:,} kept={kept} dt={dt:.2f}s"
            )

            if best is not None and kept > 0:
                chi2_arr = best["chi2"].to_numpy()
                chi2_min = np.nanmin(chi2_arr)
                wts      = np.exp(-0.5 * (chi2_arr - chi2_min))
                if np.isfinite(wts).any() and (wts.sum() > 0):
                    per_grid_mu[fam] = float(
                        (wts * best["age_gyr"].to_numpy()).sum() / wts.sum()
                    )
                    pool.append(best)
                    grid_success[fam] = True

        if all(grid_success.values()):
            break

    # GARSTEC fallback
    if (not grid_success.get("GARSTEC", False)) and ("GARSTEC" in parq):
        fam = "GARSTEC"
        pf  = parq[fam]
        cm  = colmaps[fam]
        mu_g, scanned, dt = _fit_star_garstec_simple(
            pf, cm,
            teff_o=teff_o, s_T=s_T,
            lg_seis=lg_seis, s_lg_seis=s_lg_seis
        )
        print(f"[GARSTEC-SIMPLE] row {i} scanned≈{scanned:,} age≈{mu_g:.3f} Gyr dt={dt:.2f}s")
        per_grid_mu[fam] = mu_g
        grid_success[fam] = True

    # combine all
    if pool:
        allc     = pd.concat(pool, ignore_index=True)
        chi2_arr = allc["chi2"].to_numpy()
        chi2_min = np.nanmin(chi2_arr)
        w        = np.exp(-0.5 * (chi2_arr - chi2_min))

        if np.isfinite(w).any() and (w.sum() > 0):
            mu  = float((w * allc["age_gyr"].to_numpy()).sum() / w.sum())
            var = float((w * ((allc["age_gyr"].to_numpy() - mu) ** 2)).sum() / w.sum())

            out = {
                "TICID":   r["TICID"],
                "row_i":   i,
                "age_gyr": mu,
                "age_std": math.sqrt(max(0.0, var)),
                "M_seis":  M_seis,
                "sM_seis": sM_eff,
            }
            for fam, aval in per_grid_mu.items():
                out[f"age_{fam}"] = aval
            rows.append(out)
            success = True

    if not success:
        rows.append({
            "TICID":   r["TICID"],
            "row_i":   i,
            "age_gyr": np.nan,
            "age_std": np.nan,
            "M_seis":  M_seis,
            "sM_seis": sM_eff,
        })

ages_out = pd.DataFrame(rows)
ages_out.to_csv(OUT_FILE, index=False)
print(f"\nSaved ages to: {OUT_FILE}")
try:
    from IPython.display import display
    display(ages_out.head(16))
except Exception:
    pass


**This cell is optional! it just makes some figures.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#We reload both the original catalog and the fitted ages file.
old_csv = pd.read_csv("clusterwork.csv")
new_csv = pd.read_csv("clusterwork_AGES_FIXED.csv")

# Merge on TICID 
df = pd.merge(old_csv, new_csv, on="TICID", how="inner", suffixes=('_old', '_new'))

# Identify cluster column 
cluster_col = next((c for c in df.columns if "cluster" in c.lower()), None)
if cluster_col is None:
    raise ValueError("No cluster column found in the data. Please ensure a column containing 'cluster' exists.")

# Filter to usable rows with finite ages
df = df[np.isfinite(df["age_gyr"]) & (df["age_gyr"] > 0)].copy()

# Compute cluster means
cluster_means = df.groupby(cluster_col)['age_gyr'].mean().reset_index(name='age_gyr_mean')
df = pd.merge(df, cluster_means, on=cluster_col)

#These are rough literature age ranges to overplot for MY specific clusters.
lit_ranges = {
    'alessi': (0.8, 1.45),
    'ngc': (1.1, 1.5)
}

#Change this to a path that exists on your machine; this is just an example.
outdir = "/Users/carlimankowski/research/visuals"
os.makedirs(outdir, exist_ok=True)

# my preference
sns.set(style="whitegrid")

#Boxplot of ages per cluster with literature annotations 
plt.figure(figsize=(10, 6))
sns.boxplot(x=cluster_col, y="age_gyr", data=df)
plt.title("Boxplot of Fitted Ages by Cluster")
plt.xlabel("Cluster")
plt.ylabel("Age (Gyr)")
plt.xticks(rotation=45)

clusters = df[cluster_col].unique()
for i, cl in enumerate(clusters):
    cl_lower = cl.lower()
    for key, (low, high) in lit_ranges.items():
        if key in cl_lower:
            plt.hlines(low,  i - 0.2, i + 0.2, colors='red', linestyles='--')
            plt.hlines(high, i - 0.2, i + 0.2, colors='red', linestyles='--')
            plt.text(
                i, (low + high) / 2,
                f'Lit: {low}-{high}',
                ha='center', va='center',
                color='red', rotation=90
            )

boxplot_file = os.path.join(outdir, "ages_boxplot_by_cluster.png")
plt.savefig(boxplot_file, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {boxplot_file}")

#Scatter of Individual Star Age vs Cluster Average Age
plt.figure(figsize=(12, 12))  
sns.scatterplot(
    x='age_gyr_mean',
    y='age_gyr',
    hue=cluster_col,
    data=df,
    s=150,
    alpha=0.8
)
plt.errorbar(
    df['age_gyr_mean'],
    df['age_gyr'],
    yerr=df['age_std'],
    fmt='none',
    ecolor='gray',
    alpha=0.5,
    capsize=3
)
plt.plot([0, 4], [0, 4], 'k--', label='y = x')  
plt.xlim(0, 4)
plt.ylim(0, 4)
plt.xlabel('Cluster Average Age (Gyr)')
plt.ylabel('Individual Star Age (Gyr)')
plt.title('Individual Star Ages vs Cluster Average Age')
plt.legend(title='Cluster')
scatter_file = os.path.join(outdir, "star_vs_cluster_age.png")
plt.savefig(scatter_file, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved {scatter_file}")
